# new-thyroid2: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('25_new-thyroid2_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.839,93.212,97.500,91.429,99.444,95.437,99.643,99.921,94.025
smote,98.709,96.571,94.821,98.571,98.889,98.730,98.878,99.762,97.640
bsmote,99.441,97.333,95.000,100.000,98.889,99.444,98.878,99.762,98.387
ros,99.580,98.000,96.250,100.000,99.167,99.583,99.056,99.802,98.790
adasyn,99.441,97.333,95.000,100.000,98.889,99.444,98.878,99.762,98.387
casb,99.441,97.333,95.000,100.000,98.889,99.444,98.878,99.762,98.387
casb_relaxed,99.441,97.333,95.000,100.000,98.889,99.444,98.878,99.762,98.387
casb_edited,99.441,97.333,95.000,100.000,98.889,99.444,98.878,99.762,98.387
casb_ros,99.441,97.333,95.000,100.000,98.889,99.444,98.614,99.722,98.387


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,87.161,85.007,97.500,77.143,99.444,88.294,99.235,99.841,86.084
smote,96.301,93.495,93.155,94.286,98.611,96.448,96.358,99.484,94.898
bsmote,96.903,93.597,91.905,95.714,98.333,97.024,96.193,99.405,95.250
ros,98.296,94.750,91.528,98.571,98.056,98.313,96.108,99.405,96.523
adasyn,99.019,95.417,91.528,100.000,98.056,99.028,97.321,99.683,97.218
casb,98.569,95.905,93.571,98.571,98.611,98.591,96.417,99.484,97.237
casb_relaxed,97.634,94.359,92.083,97.143,98.333,97.738,96.417,99.484,95.996
casb_edited,97.634,94.359,92.083,97.143,98.333,97.738,96.417,99.484,95.996
casb_ros,98.569,95.905,93.571,98.571,98.611,98.591,96.358,99.484,97.237


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.033,96.923,100.000,94.286,100.000,97.143,99.821,99.960,96.978
smote,95.549,95.385,100.000,91.429,100.000,95.714,99.464,99.881,95.467
bsmote,95.280,94.059,97.321,91.429,99.444,95.437,98.829,99.722,94.669
ros,95.549,95.385,100.000,91.429,100.000,95.714,99.821,99.960,95.467
adasyn,95.420,94.725,98.571,91.429,99.722,95.575,99.286,99.841,95.072
casb,94.807,94.615,100.000,90.000,100.000,95.000,99.365,99.841,94.711
casb_relaxed,95.290,94.066,97.143,91.429,99.444,95.437,99.464,99.881,94.678
casb_edited,96.032,94.835,97.143,92.857,99.444,96.151,99.286,99.841,95.434
casb_ros,95.420,94.725,98.571,91.429,99.722,95.575,99.821,99.960,95.072


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.806,93.333,100.000,88.571,100.000,94.286,99.643,99.921,93.570
smote,98.237,97.128,97.500,97.143,99.444,98.294,99.643,99.921,97.682
bsmote,98.709,96.571,94.821,98.571,98.889,98.730,99.643,99.921,97.640
ros,98.107,96.469,96.071,97.143,99.167,98.155,99.643,99.921,97.288
adasyn,98.709,96.571,94.821,98.571,98.889,98.730,99.643,99.921,97.640
casb,98.107,96.469,96.071,97.143,99.167,98.155,99.643,99.921,97.288
casb_relaxed,98.237,97.128,97.500,97.143,99.444,98.294,99.643,99.921,97.682
casb_edited,98.237,97.128,97.500,97.143,99.444,98.294,99.643,99.921,97.682
casb_ros,98.107,96.469,96.071,97.143,99.167,98.155,99.643,99.921,97.288


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.210,92.119,98.750,87.857,99.722,93.790,99.585,99.911,92.664
smote,97.199,95.645,96.369,95.357,99.236,97.297,98.586,99.762,96.422
bsmote,97.583,95.390,94.762,96.429,98.889,97.659,98.386,99.702,96.487
ros,97.883,96.151,95.962,96.786,99.097,97.941,98.657,99.772,97.017
adasyn,98.147,96.012,94.980,97.500,98.889,98.194,98.782,99.802,97.079
casb,97.731,96.081,96.161,96.429,99.167,97.798,98.576,99.752,96.906
casb_relaxed,97.650,95.722,95.432,96.429,99.028,97.728,98.600,99.762,96.686
casb_edited,97.836,95.914,95.432,96.786,99.028,97.907,98.556,99.752,96.875
casb_ros,97.884,96.108,95.804,96.786,99.097,97.941,98.609,99.772,96.996


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,99.441,ros,99.58,-0.140,LOSS
1,f1,casb,97.333,ros,98.00,-0.667,LOSS
2,precision,casb,95.000,ros,96.25,-1.250,LOSS
3,recall,casb,100.000,bsmote,100.00,0.000,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,98.569,adasyn,99.019,-0.450,LOSS
1,f1,casb,95.905,adasyn,95.417,0.488,WIN
2,precision,casb,93.571,smote,93.155,0.417,WIN
3,recall,casb,98.571,adasyn,100.000,-1.429,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,96.032,smote,95.549,0.483,WIN
1,f1,casb_edited,94.835,smote,95.385,-0.549,LOSS
2,precision,casb,100.000,smote,100.000,0.000,TIE
3,recall,casb_edited,92.857,smote,91.429,1.429,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,97.884,adasyn,98.147,-0.263,LOSS
1,f1,casb_ros,96.108,ros,96.151,-0.043,LOSS
2,precision,casb,96.161,smote,96.369,-0.208,LOSS
3,recall,casb_edited,96.786,adasyn,97.500,-0.714,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,1.000000,1.000000,0.959673,None
1,42,1,svm,smote,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
2,42,1,svm,bsmote,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
3,42,1,svm,ros,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
4,42,1,svm,adasyn,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
5,42,1,svm,casb,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
6,42,1,svm,casb_relaxed,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
7,42,1,svm,casb_edited,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.961735,0.992063,0.959673,None
8,42,1,svm,casb_ros,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,0.937925,0.988095,0.959673,None
9,42,1,knn,original,0.50,0.986013,0.933333,0.875000,1.000000,0.972222,0.986111,1.000000,1.000000,0.959673,None
